In [1]:
import numpy as np
import pandas as pd
import librosa
from scipy.signal import butter, sosfiltfilt, hilbert
from pathlib import Path
import traceback
import time
import os
import concurrent.futures
from tqdm import tqdm
from typing import Tuple, Optional, Dict

# ==============================================================================
#                           КОНФИГУРАЦИЯ
# ==============================================================================
# --- Пути ---
DATA_DIR = Path("./") # Путь к папке, где лежит train.csv и папка с аудио
AUDIO_FOLDER_NAME = "morse_dataset/morse_dataset" # Имя папки с .opus файлами
OUTPUT_FEATURE_DIR = Path("./features_1d_precomputed") # Папка для сохранения .npy файлов
TRAIN_CSV_PATH = DATA_DIR / "train.csv"

# --- Параметры Аудио и Поиска Частот ---
TARGET_SR = 8000
N_FFT_FIND = 16384 # Должно совпадать с CONFIG
HOP_LEN_FIND = 1024 # Должно совпадать с CONFIG
ENERGY_THRESH_PERC_FIND = 30.0
MASK_RELATIVE_WIDTH_FIND = 0.4

# --- Параметры Извлечения Мульти-полосных Огибающих ---
NUM_SIDE_BANDS = 3
FREQ_STEP_HZ = 5.0
EXTRACTION_BW_HZ = 30.0
FILTER_ORDER = 5

# --- Параметры Параллельной Обработки ---
# Используй os.cpu_count() - 1 или 2, чтобы оставить ресурсы для системы
# Или установи конкретное число, например 4 или 8
MAX_WORKERS = 1
# print(MAX_WORKERS)
# --- Имена столбцов ---
FILE_ID_COLUMN = 'id' # Имя столбца с ID файла в CSV

# ==============================================================================
#                  ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ (из ячеек 1 и 2)
# ==============================================================================

# --- Полосовой фильтр ---
def bandpass_filter(data: np.ndarray, lowcut: float, highcut: float, fs: int, order: int = 4) -> np.ndarray:
    nyq = 0.5 * fs; low = lowcut / nyq; high = highcut / nyq
    if low <= 0: low = 0.001
    if high >= 1: high = 0.999
    if low >= high: return data # Ошибка уже не логируется здесь
    try:
        sos = butter(order, [low, high], btype='band', output='sos')
        return sosfiltfilt(sos, data)
    except Exception: # Ловим любую ошибку фильтрации
        return data # Возвращаем оригинал в случае ошибки

# --- Поиск центральных частот ---
def find_frequencies_large_nfft(y: np.ndarray, sr: int, n_fft_large: int, hop_len_large: int,
                                energy_thresh_percentile: float, mask_relative_width: float
                               ) -> Tuple[Optional[float], Optional[float]]:
    f_carrier = None; f_harmonic3 = None
    try:
        if y is None or len(y) < n_fft_large: return None, None
        if hop_len_large >= n_fft_large: hop_len_large = n_fft_large // 4
        S_complex = librosa.stft(y, n_fft=n_fft_large, hop_length=hop_len_large)
        S_amp = np.abs(S_complex); n_freq_bins, n_frames = S_amp.shape
        if n_frames < 5: return None, None
        freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft_large)
        rms_per_frame = librosa.feature.rms(S=S_amp, frame_length=n_fft_large, hop_length=hop_len_large)[0]
        if len(rms_per_frame) != n_frames: rms_per_frame = np.resize(rms_per_frame, n_frames)
        energy_threshold = np.percentile(rms_per_frame, energy_thresh_percentile)
        active_frames_indices = np.where(rms_per_frame > energy_threshold)[0]
        if len(active_frames_indices) < 5: return None, None
        peak_indices_f1 = np.argmax(S_amp[:, active_frames_indices], axis=0); peak_freqs_f1 = freqs[peak_indices_f1]
        f_carrier = np.median(peak_freqs_f1)
        if f_carrier is None or not np.isfinite(f_carrier): return None, None
        mask_center_freq = f_carrier; mask_half_width = mask_relative_width * f_carrier
        mask_freq_low = mask_center_freq - mask_half_width; mask_freq_high = mask_center_freq + mask_half_width
        mask_indices = np.where((freqs >= mask_freq_low) & (freqs <= mask_freq_high))[0]
        S_amp_masked = S_amp.copy()
        if len(mask_indices) > 0: S_amp_masked[mask_indices[:, np.newaxis], active_frames_indices] = 0.0
        peak_indices_f3 = np.argmax(S_amp_masked[:, active_frames_indices], axis=0); peak_freqs_f3 = freqs[peak_indices_f3]
        f_harmonic3 = np.median(peak_freqs_f3)
        if f_harmonic3 is None or not np.isfinite(f_harmonic3): return f_carrier, None
        return f_carrier, f_harmonic3
    except Exception: return None, None # Возвращаем None при любой ошибке

# --- Извлечение мульти-полосных огибающих ---
def extract_multi_band_envelopes(y: np.ndarray, sr: int, f1: Optional[float], f3: Optional[float],
                                 num_side_bands: int, freq_step_hz: float, extraction_bw_hz: float, filter_order: int
                                ) -> Dict[str, Optional[np.ndarray]]:
    envelopes: Dict[str, Optional[np.ndarray]] = {}
    offsets_hz = np.arange(-num_side_bands, num_side_bands + 1) * freq_step_hz
    expected_len = len(y) # Ожидаемая длина равна исходному сигналу

    for base_freq, prefix in [(f1, "f1"), (f3, "f3")]:
        if base_freq is None:
            for offset in offsets_hz:
                 band_name = f"{prefix}_{int(offset):+d}Hz"
                 envelopes[band_name] = np.zeros(expected_len, dtype=np.float32) # Заполняем нулями
            continue

        for offset in offsets_hz:
            target_freq = base_freq + offset
            band_name = f"{prefix}_{int(offset):+d}Hz"
            lowcut = target_freq - extraction_bw_hz / 2.0
            highcut = target_freq + extraction_bw_hz / 2.0

            if lowcut < 0 or highcut > sr / 2.0:
                envelopes[band_name] = np.zeros(expected_len, dtype=np.float32)
                continue
            try:
                y_filtered = bandpass_filter(y, lowcut, highcut, sr, filter_order)
                if np.array_equal(y_filtered, y) or np.allclose(y_filtered, 0):
                    envelopes[band_name] = np.zeros(expected_len, dtype=np.float32)
                    continue
                analytic_signal = hilbert(y_filtered)
                envelope = np.abs(analytic_signal)
                # Простое выравнивание до исходной длины (на всякий случай)
                if len(envelope) != expected_len:
                    if len(envelope) > expected_len: envelope = envelope[:expected_len]
                    else: envelope = np.pad(envelope, (0, expected_len - len(envelope)))
                envelopes[band_name] = envelope.astype(np.float32)
            except Exception:
                envelopes[band_name] = np.zeros(expected_len, dtype=np.float32) # Нули при ошибке

    return envelopes

# ==============================================================================
#                  ФУНКЦИЯ ОБРАБОТКИ ОДНОГО ФАЙЛА (Исправленная v14.1)
# ==============================================================================

def process_file(args: Tuple[str, Path, Path]) -> Tuple[str, bool, Optional[str]]:
    """
    Обрабатывает один аудиофайл: загрузка, поиск частот, извлечение 14 полос, сохранение.
    Исправлено формирование пути к аудиофайлу и выходному файлу.
    """
    file_id_str, audio_folder_path, output_feature_dir = args # file_id может быть с расширением или без
    file_id_path = Path(file_id_str) # Преобразуем в Path для работы с суффиксом

    # Имя выходного файла: берем основу имени (stem) и добавляем .npy
    output_filename = file_id_path.stem + ".npy"
    output_path = output_feature_dir / output_filename

    # Формируем путь к аудио, гарантируя суффикс .opus
    # Path(...).with_suffix('.opus') заменит существующий суффикс или добавит новый
    audio_filename = file_id_path.with_suffix('.opus').name
    audio_path = audio_folder_path / audio_filename

    # --- Проверка, не обработан ли файл уже ---
    if output_path.exists():
        # Возвращаем исходный file_id_str для консистентности
        return file_id_str, True, f"Skipped (already exists): {output_path.name}"

    # --- Загрузка и подготовка аудио ---
    try:
        # Используем исправленный audio_path
        y, sr_orig = librosa.load(audio_path, sr=None, mono=True)
        if y is None or len(y) == 0: return file_id_str, False, "Error: Empty signal loaded"
        if sr_orig != TARGET_SR:
            y = librosa.resample(y=y, orig_sr=sr_orig, target_sr=TARGET_SR)
            if y is None or len(y) == 0: return file_id_str, False, "Error: Empty signal after resampling"
        sr = TARGET_SR
        y = librosa.util.normalize(y)
    except FileNotFoundError:
        # Выводим путь, который не был найден
        return file_id_str, False, f"Error: Audio file not found at {audio_path}"
    except Exception as e_load:
        return file_id_str, False, f"Error loading/processing audio: {e_load}"

    # --- Поиск частот ---
    try:
        f1, f3 = find_frequencies_large_nfft(y, sr, N_FFT_FIND, HOP_LEN_FIND,
                                           ENERGY_THRESH_PERC_FIND, MASK_RELATIVE_WIDTH_FIND)
        if f1 is None:
            return file_id_str, False, "Error: Could not find f1 frequency"
    except Exception as e_find:
        return file_id_str, False, f"Error during frequency finding: {e_find}"

    # --- Извлечение огибающих ---
    try:
        multi_band_envelopes = extract_multi_band_envelopes(
            y, sr, f1, f3, NUM_SIDE_BANDS, FREQ_STEP_HZ, EXTRACTION_BW_HZ, FILTER_ORDER
        )
    except Exception as e_extract:
        return file_id_str, False, f"Error during envelope extraction: {e_extract}"

    # --- Сборка признаков в массив (14, T) ---
    feature_list = []
    expected_len = len(y)
    offsets_hz = np.arange(-NUM_SIDE_BANDS, NUM_SIDE_BANDS + 1) * FREQ_STEP_HZ
    for prefix in ["f1", "f3"]:
        for offset in offsets_hz:
            band_name = f"{prefix}_{int(offset):+d}Hz"
            envelope = multi_band_envelopes.get(band_name)
            # Проверяем, что огибающая не None И имеет правильную длину
            if envelope is not None and len(envelope) == expected_len:
                feature_list.append(envelope)
            else:
                # Если None или неверная длина, добавляем нули
                feature_list.append(np.zeros(expected_len, dtype=np.float32))

    # Проверяем, что собрали ровно 14 полос
    expected_num_bands = 2 * (2 * NUM_SIDE_BANDS + 1)
    if len(feature_list) != expected_num_bands:
         return file_id_str, False, f"Error: Incorrect number of bands assembled ({len(feature_list)}), expected {expected_num_bands}"

    features_np = np.stack(feature_list, axis=0).astype(np.float32) # (14, T)

    # --- Сохранение результата ---
    try:
        # Используем исправленный output_path
        np.save(output_path, features_np)
        # Возвращаем исходный file_id_str и имя сохраненного файла
        return file_id_str, True, output_path.name
    except Exception as e_save:
        return file_id_str, False, f"Error saving .npy file: {e_save}"

# ==============================================================================
#                            ОСНОВНОЙ БЛОК
# ==============================================================================
if __name__ == "__main__":
    print("--- Запуск Скрипта Предварительного Расчета 1D Признаков ---")
    start_time_script = time.time()

    # --- Проверка и создание папок ---
    audio_folder = DATA_DIR / AUDIO_FOLDER_NAME
    if not audio_folder.is_dir():
        print(f"❌ ОШИБКА: Папка с аудио не найдена: {audio_folder}")
        exit()
    if not TRAIN_CSV_PATH.is_file():
        print(f"❌ ОШИБКА: Файл {TRAIN_CSV_PATH} не найден.")
        exit()

    OUTPUT_FEATURE_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Папка для сохранения признаков: {OUTPUT_FEATURE_DIR.resolve()}")

    # --- Загрузка CSV ---
    try:
        train_df = pd.read_csv(TRAIN_CSV_PATH)
        print(f"Загружен {TRAIN_CSV_PATH}: {len(train_df)} записей.")
        if FILE_ID_COLUMN not in train_df.columns:
            print(f"❌ ОШИБКА: Столбец '{FILE_ID_COLUMN}' не найден в {TRAIN_CSV_PATH}")
            exit()
    except Exception as e_csv:
        print(f"❌ ОШИБКА при чтении CSV: {e_csv}")
        exit()

    # --- Подготовка аргументов для параллельной обработки ---
    tasks = []
    for file_id in train_df[FILE_ID_COLUMN]:
        tasks.append((str(file_id), audio_folder, OUTPUT_FEATURE_DIR))

    # --- Параллельная обработка ---
    print("\nЗапуск ПОСЛЕДОВАТЕЛЬНОЙ обработки (для отладки)...")
    results = []
    success_count = 0
    fail_count = 0
    skip_count = 0
    failed_files = []
    for task_args in tqdm(tasks, desc="Обработка файлов (послед.)"):
        result_tuple = process_file(task_args) # Вызываем напрямую
        results.append(result_tuple)
        # Можно сразу проверять результат для вывода ошибки
        file_id, success, message = result_tuple
        if not success:
            print(f"\n -> Ошибка при обработке {file_id}: {message}")
            # Можно добавить break здесь, если хочешь остановиться на первой ошибке
            # break

    print("\nОбработка завершена. Подсчет результатов...")
    for file_id, success, message in results:
        if success:
            if "Skipped" in message:
                skip_count += 1
            else:
                success_count += 1
        else:
            fail_count += 1
            failed_files.append((file_id, message))
            # Печатаем ошибку для первых нескольких неудач
            if fail_count <= 20:
                 print(f"  -> Ошибка для {file_id}: {message}")
            elif fail_count == 21:
                 print("  -> (Дальнейшие ошибки не будут выводиться)")


    # --- Вывод статистики ---
    end_time_script = time.time()
    total_duration = end_time_script - start_time_script
    print("\n--- Статистика Предварительного Расчета ---")
    print(f"Всего файлов в CSV: {len(train_df)}")
    print(f"Успешно обработано и сохранено: {success_count}")
    print(f"Пропущено (уже существовали): {skip_count}")
    print(f"Не удалось обработать: {fail_count}")
    print(f"Общее время выполнения: {total_duration:.2f} сек ({total_duration/60:.2f} мин)")

    if failed_files:
        print("\nСписок файлов, которые не удалось обработать:")
        for fid, msg in failed_files[:min(len(failed_files), 50)]: # Показываем не более 50
            print(f"  - {fid}: {msg}")
        if len(failed_files) > 50:
            print(f"  ... и еще {len(failed_files) - 50} файлов.")

    print("\n--- Предварительный расчет завершен. ---")
    print(f"Теперь можно обновить MorseDataset1D для загрузки файлов из: {OUTPUT_FEATURE_DIR}")

--- Запуск Скрипта Предварительного Расчета 1D Признаков ---
Папка для сохранения признаков: C:\Users\vasja\OneDrive\Рабочий стол\Morse_dev\MorseAudioDecoder\features_1d_precomputed
Загружен train.csv: 30000 записей.

Запуск ПОСЛЕДОВАТЕЛЬНОЙ обработки (для отладки)...


Обработка файлов (послед.):  62%|██████▏   | 18558/30000 [00:01<00:00, 14288.70it/s]


KeyboardInterrupt: 

In [10]:
# Ячейка 13.3 (Исправленная v2): Анализ с Фильтром Duration

import optuna
import pandas as pd
import traceback
import time
import numpy as np

print("--- Анализ Результатов Optuna HPO (Многокритериальный) ---")

# --- Укажите study_name и storage_name для МНОГОКРИТЕРИАЛЬНОГО исследования ---
try:
    # Пытаемся взять из переменных текущего запуска
    # _ = study_name_multi # Закомментировано, т.к. может не существовать при отдельном запуске
    # _ = storage_name_multi
    # study_name = study_name_multi
    # storage_name = storage_name_multi
    # --- ЗАДАЙТЕ ВРУЧНУЮ ЗДЕСЬ ---
    study_name = "morse-1d-multiobj-hpo-v5.0-20250427_1654" # <-- УБЕДИТЕСЬ, ЧТО ИМЯ ВЕРНОЕ
    storage_name = f"sqlite:///{study_name}.db"
    # -----------------------------
    print(f"Используются study_name='{study_name}' и storage='{storage_name}' для MultiObj HPO.")
except NameError:
    print("!!! Переменные study_name/storage_name не найдены. Задайте их вручную в коде! !!!")
    raise

print(f"Загрузка данных исследования '{study_name}' из '{storage_name}'...")

try:
    # Загружаем исследование
    loaded_study = optuna.load_study(
        study_name=study_name,
        storage=storage_name
    )

    print(f"\nИсследование '{loaded_study.study_name}' успешно загружено.")
    print(f"Направления оптимизации: {loaded_study.directions}")

    # --- Статистика по Пробам ---
    completed_trials = [t for t in loaded_study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    pruned_trials = [t for t in loaded_study.trials if t.state == optuna.trial.TrialState.PRUNED]
    failed_trials = [t for t in loaded_study.trials if t.state == optuna.trial.TrialState.FAIL]
    running_trials = [t for t in loaded_study.trials if t.state == optuna.trial.TrialState.RUNNING]
    print(f"\n--- Статистика по Пробам ---")
    print(f"Всего проб: {len(loaded_study.trials)}")
    print(f"  COMPLETE: {len(completed_trials)}, PRUNED: {len(pruned_trials)}, FAIL: {len(failed_trials)}, RUNNING: {len(running_trials)}")

    # --- Парето-фронт ---
    pareto_trials = []
    if completed_trials:
        try:
            pareto_trials = loaded_study.best_trials
            print(f"\n--- Парето-оптимальные Пробы ({len(pareto_trials)}) ---")
            pareto_trials.sort(key=lambda t: t.values[0] if t.values else float('inf'))

            for i, trial in enumerate(pareto_trials):
                print(f"\n  Проба #{trial.number} (Pareto {i+1})")
                print(f"    Значения целей:")
                if trial.values and len(trial.values) == 3:
                    print(f"      1. BestLev:      {trial.values[0]:.4f}")
                    print(f"      2. SumLevLast4:  {trial.values[1]:.4f}")
                    print(f"      3. LossRatio:    {trial.values[2]:.4f}")
                else: print("      (Значения целей недоступны)")
                print(f"    Параметры:")
                for key, value in trial.params.items(): print(f"      {key}: {value}")
        except Exception as e_pareto:
             print(f"\nОшибка при получении/обработке Парето-фронта: {e_pareto}")
    else:
        print("\n--- Парето-фронт ---")
        print("  Нет завершенных проб для анализа.")

    # --- Таблица Результатов (DataFrame) - ПОЛНАЯ ВЕРСИЯ (Фильтр по Duration) ---
    print("\n--- Таблица Всех Проб (ПОЛНАЯ, duration > 30 сек) ---")
    # Используем вложенный try-except для Pandas операций, чтобы точно сбросить опции
    original_max_rows, original_max_cols, original_width = None, None, None
    try:
        df_trials = loaded_study.trials_dataframe()

        # Фильтр по Duration
        min_duration_threshold = pd.Timedelta(seconds=30)
        df_filtered_by_duration = df_trials[df_trials['duration'] > min_duration_threshold].copy()
        print(f"Фильтр: duration > {min_duration_threshold}. Показано {len(df_filtered_by_duration)} из {len(df_trials)} проб.")
    
        # Определяем столбцы для показа
        cols_to_show = ['number', 'values_0', 'values_1', 'values_2', 'state', 'duration']
        param_keys_to_add = []
        if pareto_trials: param_keys_to_add = pareto_trials[0].params.keys()
        elif loaded_study.trials: param_keys_to_add = loaded_study.trials[0].params.keys()
        cols_to_show.extend([f'params_{p}' for p in param_keys_to_add])
        cols_to_show = [c for c in cols_to_show if c in df_filtered_by_duration.columns]
        
        # Устанавливаем опции для полного вывода
        original_max_rows = pd.get_option('display.max_rows')
        original_max_cols = pd.get_option('display.max_columns')
        original_width = pd.get_option('display.width')
        print("Установка временных опций Pandas для полного вывода...")
        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_columns', None)
        pd.set_option('display.width', 2000)

        # Печатаем ОТФИЛЬТРОВАННЫЙ DataFrame
        print("Столбцы целей: values_0=BestLev, values_1=SumLevLast4, values_2=LossRatio")
        print(df_filtered_by_duration[cols_to_show].sort_values(by='number'))

    except Exception as e_df:
        print(f"Ошибка при создании или выводе DataFrame: {e_df}")
        traceback.print_exc(limit=1)
    finally:
        # Возвращаем опции к исходным значениям, даже если была ошибка
        if original_max_rows is not None:
            print("Восстановление стандартных опций Pandas...")
            pd.set_option('display.max_rows', original_max_rows)
            pd.set_option('display.max_columns', original_max_cols)
            pd.set_option('display.width', original_width)
            print("Опции Pandas восстановлены.")

    # --- Визуализация (если установлен plotly) ---
    print("\n--- Визуализация Результатов ---")
    if optuna.visualization.is_available():
        print("Plotly найден. Построение графиков...")
        try:
            if pareto_trials:
                 fig_pareto = optuna.visualization.plot_pareto_front(loaded_study, target_names=["BestLev", "SumLevLast4", "LossRatio"])
                 fig_pareto.show()
            else: print("Нет Парето-оптимальных проб для визуализации.")
        except ImportError as e_plotly: print(f"Ошибка импорта Plotly: {e_plotly}")
        except Exception as e_vis: print(f"Ошибка при построении графиков: {e_vis}")
    else: print("Установите Plotly (`pip install plotly kaleido`) для визуализации.")

except FileNotFoundError: print(f"❌ ОШИБКА: Файл базы данных Optuna не найден: {storage_name}")
except Exception as e: print(f"❌ Непредвиденная ошибка: {e}"); traceback.print_exc()

print("\n--- Анализ Завершен ---")

--- Анализ Результатов Optuna HPO (Многокритериальный) ---
Используются study_name='morse-1d-multiobj-hpo-v5.0-20250427_1654' и storage='sqlite:///morse-1d-multiobj-hpo-v5.0-20250427_1654.db' для MultiObj HPO.
Загрузка данных исследования 'morse-1d-multiobj-hpo-v5.0-20250427_1654' из 'sqlite:///morse-1d-multiobj-hpo-v5.0-20250427_1654.db'...

Исследование 'morse-1d-multiobj-hpo-v5.0-20250427_1654' успешно загружено.
Направления оптимизации: [<StudyDirection.MINIMIZE: 1>, <StudyDirection.MINIMIZE: 1>, <StudyDirection.MINIMIZE: 1>]

--- Статистика по Пробам ---
Всего проб: 1716
  COMPLETE: 101, PRUNED: 1612, FAIL: 0, RUNNING: 3

--- Парето-оптимальные Пробы (11) ---

  Проба #86 (Pareto 1)
    Значения целей:
      1. BestLev:      0.2200
      2. SumLevLast4:  0.9800
      3. LossRatio:    2.7296
    Параметры:
      pool_stride_combo_idx: 14
      channel_combo_idx: 15
      k1_rule: s+1
      k2_rule: s*2+1

  Проба #39 (Pareto 2)
    Значения целей:
      1. BestLev:      0.2367
    


--- Анализ Завершен ---
